**Install required libraries**

In [ ]:
!pip install -q transformers datasets scikit-learn pandas seaborn matplotlib tqdm

# **Dataset**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/Project1 (Sentiment Analysis on Amazon Product Reviews)/Reviews.csv", on_bad_lines='skip')
df = df[['Text','Score']].dropna()
df.rename(columns={'Text':'text','Score':'score'}, inplace=True)

# Map numeric scores to sentiment labels
def map_label(s):
    if s <= 2:
        return 'negative'
    elif s == 3:
        return 'neutral'
    else:
        return 'positive'

df['label'] = df['score'].apply(map_label)

# Display label distribution
print(df['label'].value_counts())
sns.countplot(data=df, x='label')
plt.title("Label Distribution")
plt.show()

 **Dataset İnfo**

- 568,000 Amazon product reviews  
- 443777 Positive, 82037 Negative, 42640 Neutral Review
- Source: Amazon open data  
- Score used as ground truth (1–5 → sentiment)  

# **Dataset Preprocessing**

In [ ]:
import re
from sklearn.model_selection import train_test_split

# Text Cleaning Function
# Removes URLs, special characters, and converts to lowercase

def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^A-Za-z0-9(),!?\'\`]", " ", text)
    text = text.lower()
    return text.strip()

df['clean_text'] = df['text'].apply(clean_text)

# Split into train/validation/test (80/10/10)
train_val, test = train_test_split(df, test_size=0.1, stratify=df['label'], random_state=42)
train, val = train_test_split(train_val, test_size=0.1111, stratify=train_val['label'], random_state=42)

print(f"Train: {len(train)}, Validation: {len(val)}, Test: {len(test)}")

**Preprocessing Steps**

1. Removed URLs and special characters  
2. Converted text to lowercase  
3. Split dataset into train (80%)(454768), validation (10%)(56480), test (10%)(56846)

In [ ]:
df.head()

# **Method 0: Naive Bayes**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

nb_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2))),
    ('clf', MultinomialNB())
])

nb_model.fit(train['clean_text'], train['label'])
nb_preds = nb_model.predict(val['clean_text'])

nb_acc = accuracy_score(val['label'], nb_preds)
nb_f1 = f1_score(val['label'], nb_preds, average='macro')

print("Naive Bayes Results")
print("Accuracy:", nb_acc)
print("F1 Score:", nb_f1)
print(classification_report(val['label'], nb_preds))

**Naive Bayes Results**
- Accuracy ≈ 0.86  
- F1 Score ≈ 0.61  
*Interpretation:* Good baseline performance but struggles with neutral reviews.

# **Method 1: Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

lr_model.fit(train['clean_text'], train['label'])
lr_preds = lr_model.predict(val['clean_text'])

lr_acc = accuracy_score(val['label'], lr_preds)
lr_f1 = f1_score(val['label'], lr_preds, average='macro')

print("Logistic Regression Results")
print("Accuracy:", lr_acc)
print("F1 Score:", lr_f1)
print(classification_report(val['label'], lr_preds))

**Logistic Regression Results**
- Accuracy ≈ 0.86  
- F1 Score ≈ 0.74  
*Interpretation:* Handles text patterns better than Naive Bayes.

# **Method 2: DistilBERT**

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np


# Pretrained model fine-tuned for sentiment classification.
# DistilBERT is a smaller, faster version of BERT.


label_list = ['negative','neutral','positive']
label2id = {v:k for k,v in enumerate(label_list)}
id2label = {k:v for k,v in enumerate(label_list)}

train_ds = Dataset.from_pandas(train[['clean_text','label']])
val_ds = Dataset.from_pandas(val[['clean_text','label']])

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def encode_data(example):
    return tokenizer(example['clean_text'], truncation=True, padding='max_length', max_length=256)

def encode_label(example):
    example['labels'] = label_list.index(example['label'])
    return example

train_ds = train_ds.map(encode_label).map(encode_data, batched=True)
val_ds = val_ds.map(encode_label).map(encode_data, batched=True)

cols_to_remove = ['clean_text','label']
train_ds = train_ds.remove_columns(cols_to_remove)
val_ds = val_ds.remove_columns(cols_to_remove)
train_ds.set_format("torch")
val_ds.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,

)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
results = trainer.evaluate(val_ds)
print(results)

**DistilBERT Results**

- Accuracy ≈ 0.92  
- F1 Score ≈ 0.80  
*Interpretation:* Transformer model captures contextual meaning and provides the best performance.

# **Results and Conclusions**

In [ ]:
results_df = pd.DataFrame({
    "Model": ["Naive Bayes", "Logistic Regression", "DistilBERT"],
    "Accuracy": [nb_acc, lr_acc, results["eval_accuracy"]],
    "F1-Score": [nb_f1, lr_f1, results["eval_f1"]]
})
print(results_df)
sns.barplot(data=results_df.melt(id_vars="Model"), x="Model", y="value", hue="variable")
plt.title("Model Performance Comparison")
plt.show()